# Homelab Sparkice smoke test
Run under JupyterHub. This test **creates data** in the `home-iceberg` bucket using the new `ice` catalog.


In [ ]:
import os, boto3
s3 = boto3.client('s3', endpoint_url='http://rustfs:9000', region_name=os.environ.get('AWS_REGION', 'us-east-1'))
s3.head_bucket(Bucket='home-iceberg')  # needs only access to this bucket
print('Bucket home-iceberg is reachable')


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('homelab-sparkice-smoke').getOrCreate()
print('Spark:', spark.version)
print('Master:', spark.sparkContext.master)
print('Executors:', spark.sparkContext._jsc.sc().getExecutorMemoryStatus().size())
spark.range(100).groupBy().count().show()


In [ ]:
spark.sql('CREATE NAMESPACE IF NOT EXISTS ice.homelab')
spark.sql("""CREATE TABLE IF NOT EXISTS ice.homelab.smoke (
    id BIGINT, source STRING
) USING iceberg""")
spark.sql("INSERT INTO ice.homelab.smoke VALUES (1, 'hp-server')")
spark.sql('SELECT * FROM ice.homelab.smoke ORDER BY id').show()
spark.sql('SHOW TABLES IN ice.homelab').show()


In [ ]:
spark.range(10).write.mode('overwrite').parquet('s3a://home-iceberg/smoke/parquet_test')
print('Parquet rows:', spark.read.parquet('s3a://home-iceberg/smoke/parquet_test').count())
# For Spark UI: spark_ui_link()  (if startup helper is installed)
